# Water Meter YOLO Training Notebook

## What to zip before uploading

You only need two folders. Create a zip like this:

```
water_meter_project.zip
└── yolo_labeler-main/
    ├── raw_dataset/
    │   ├── images/   ← all your labeled .jpg/.jpeg/.png files
    │   └── labels/   ← matching .txt YOLO label files
    └── scripts/
        ├── 02_prepare_dataset.py
        ├── 03_train.py
        ├── 04_predict.py
        └── 05_retrain.py
```

Everything else (dataset/, training_runs/, prediction_outputs/) is generated by this notebook.

## Two upload options — pick ONE in Step 1:
- **Option A** — Upload zip directly to Colab (simplest, no Drive needed)
- **Option B** — Upload zip to Google Drive first, then let the notebook pull it

## Speed optimisations applied
| Setting | Value | Effect |
|---|---|---|
| Model | yolov8n.pt | Nano — fastest, ~6 MB |
| imgsz | 416 | ~2× faster than 640 |
| batch | 32 | Fills T4 VRAM |
| cache | True | RAM-caches images after epoch 1 |
| amp | True | FP16 — ~1.5× GPU speedup |
| patience | 20 | Early stop if no improvement |
| optimizer | AdamW | Faster convergence than SGD |

## Step 0 — Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml

## Step 1 — Upload your zip

### Option A — Direct upload to Colab (recommended for first-time use)
Run the cell below. A file picker will appear. Select your `water_meter_project.zip`.
Upload speed depends on your internet connection (~600 images ≈ 50–200 MB).

### Option B — Upload to Google Drive instead
Skip this cell and run the **Option B** cell further below.

In [ ]:
# ── OPTION A: Direct upload ──────────────────────────────────────────────────
import os, zipfile, shutil
from google.colab import files

print("A file picker will open. Select your water_meter_project.zip ...")
uploaded = files.upload()   # opens the file picker

zip_filename = list(uploaded.keys())[0]
zip_path = f"/content/{zip_filename}"
print(f"\nUploaded: {zip_filename} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

# Extract to /content/
print("Extracting ...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/")

# Auto-detect the extracted project root (handles any zip structure)
LOCAL_ROOT = None
for entry in os.listdir("/content/"):
    candidate = f"/content/{entry}"
    if os.path.isdir(candidate) and os.path.isdir(f"{candidate}/raw_dataset"):
        LOCAL_ROOT = candidate
        break

if LOCAL_ROOT is None:
    # Fallback: maybe raw_dataset is directly at /content/raw_dataset
    if os.path.isdir("/content/raw_dataset"):
        LOCAL_ROOT = "/content"
    else:
        raise RuntimeError(
            "Could not find raw_dataset/ after extraction.\n"
            "Make sure your zip contains yolo_labeler-main/raw_dataset/ at the top level."
        )

print(f"\nProject root detected: {LOCAL_ROOT}")

# Verify required folders
for folder in ["raw_dataset/images", "raw_dataset/labels", "scripts"]:
    path = os.path.join(LOCAL_ROOT, folder)
    assert os.path.isdir(path), f"Missing folder: {path}"
    count = len(os.listdir(path))
    print(f"  {folder}: {count} files")

# Clean up the zip to free space
os.remove(zip_path)
print("\nReady. Proceed to Step 2.")

In [ ]:
# ── OPTION B: Upload zip to Google Drive first, then pull it here ─────────────
# Instructions:
#   1. Go to drive.google.com
#   2. Upload water_meter_project.zip anywhere (e.g. My Drive root)
#   3. Set DRIVE_ZIP_PATH below to the full path of the zip in your Drive
#   4. Run this cell (skip the Option A cell above)

import os, zipfile, shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# ── Edit this path ──
DRIVE_ZIP_PATH = "/content/drive/MyDrive/water_meter_project.zip"  # @param {type:"string"}

assert os.path.exists(DRIVE_ZIP_PATH), f"Zip not found at: {DRIVE_ZIP_PATH}"
print(f"Found zip: {DRIVE_ZIP_PATH} ({os.path.getsize(DRIVE_ZIP_PATH) / 1e6:.1f} MB)")

# Copy zip to local SSD first (much faster extraction)
local_zip = "/content/water_meter_project.zip"
print("Copying from Drive to local SSD ...")
shutil.copy2(DRIVE_ZIP_PATH, local_zip)

print("Extracting ...")
with zipfile.ZipFile(local_zip, 'r') as z:
    z.extractall("/content/")

# Auto-detect the extracted project root
LOCAL_ROOT = None
for entry in os.listdir("/content/"):
    candidate = f"/content/{entry}"
    if os.path.isdir(candidate) and os.path.isdir(f"{candidate}/raw_dataset"):
        LOCAL_ROOT = candidate
        break

if LOCAL_ROOT is None:
    if os.path.isdir("/content/raw_dataset"):
        LOCAL_ROOT = "/content"
    else:
        raise RuntimeError(
            "Could not find raw_dataset/ after extraction.\n"
            "Make sure your zip contains yolo_labeler-main/raw_dataset/ at the top level."
        )

print(f"\nProject root detected: {LOCAL_ROOT}")

# Verify required folders
for folder in ["raw_dataset/images", "raw_dataset/labels", "scripts"]:
    path = os.path.join(LOCAL_ROOT, folder)
    assert os.path.isdir(path), f"Missing folder: {path}"
    count = len(os.listdir(path))
    print(f"  {folder}: {count} files")

os.remove(local_zip)
print("\nReady. Proceed to Step 2.")

## Step 2 — Prepare dataset (validate + split into train / val / test)

In [ ]:
import subprocess, sys

RAW_DATASET    = os.path.join(LOCAL_ROOT, "raw_dataset")
DATASET_OUT    = os.path.join(LOCAL_ROOT, "dataset")
PREPARE_SCRIPT = os.path.join(LOCAL_ROOT, "scripts", "02_prepare_dataset.py")

result = subprocess.run(
    [
        sys.executable, PREPARE_SCRIPT,
        RAW_DATASET, DATASET_OUT,
        "--train", "0.70",
        "--val",   "0.20",
        "--test",  "0.10",
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Dataset preparation failed — see STDERR above.")

## Step 3 — Fix data.yaml path for Colab

`02_prepare_dataset.py` writes the absolute path of the machine that created it (your Windows path).
This cell overwrites it with the correct Colab path so YOLO can find the images.

In [ ]:
import yaml

DATA_YAML = os.path.join(DATASET_OUT, "data.yaml")

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = DATASET_OUT

with open(DATA_YAML, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print("data.yaml updated:")
with open(DATA_YAML) as f:
    print(f.read())

## Step 4 — Verify GPU

If no GPU is shown: **Runtime → Change runtime type → T4 GPU → Save**, then re-run from Step 0.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU.")

## Step 5 — Train

Expected time on Colab T4 GPU with 600 images: **~15–25 minutes** for 50 epochs.
Early stopping (`patience=20`) will cut it short if the model converges faster.

In [ ]:
from ultralytics import YOLO

TRAINING_RUNS = os.path.join(LOCAL_ROOT, "training_runs")
os.makedirs(TRAINING_RUNS, exist_ok=True)

model = YOLO("yolov8n.pt")  # ~6 MB download on first run

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=416,
    batch=32,
    workers=4,
    cache=True,
    amp=True,
    patience=20,
    optimizer="AdamW",
    cos_lr=True,
    project=TRAINING_RUNS,
    name="water_meter_yolov8n",
    exist_ok=True,
)

print("\nTraining complete.")
print(f"Best model saved at: {results.save_dir}/weights/best.pt")

## Step 6 — Evaluate on test set

In [ ]:
import glob

best_models = glob.glob(os.path.join(TRAINING_RUNS, "*/weights/best.pt"))
best_model_path = max(best_models, key=os.path.getmtime)
print(f"Evaluating: {best_model_path}")

model = YOLO(best_model_path)
metrics = model.val(data=DATA_YAML, split="test", imgsz=416)

print(f"\nmAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## Step 7 — Predict on test images and visualise

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

TEST_IMAGES_DIR = os.path.join(DATASET_OUT, "images", "test")
PREDICT_OUT     = os.path.join(LOCAL_ROOT, "prediction_outputs")

model = YOLO(best_model_path)
model.predict(
    source=TEST_IMAGES_DIR,
    imgsz=416,
    conf=0.25,
    save=True,
    save_txt=True,
    save_conf=True,
    show_labels=True,
    show_conf=False,
    project=PREDICT_OUT,
    name="test_predictions",
    exist_ok=True,
)

# Show first 6 annotated results inline
annotated_dir = os.path.join(PREDICT_OUT, "test_predictions")
image_paths = sorted(Path(annotated_dir).glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flat, image_paths):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\nAll predictions saved to: {annotated_dir}")

## Step 8 — Save best model to Google Drive

Colab sessions reset after a few hours — always save your model to Drive before closing.

In [ ]:
# If you used Option A (direct upload), Drive may not be mounted yet — mount it now
try:
    from google.colab import drive as _drive
    _drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# ── Set where in your Drive to save the model ──
DRIVE_SAVE_DIR = "/content/drive/MyDrive/water_meter_model"  # @param {type:"string"}

best_models = glob.glob(os.path.join(TRAINING_RUNS, "*/weights/best.pt"))
final_best  = max(best_models, key=os.path.getmtime)

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
dest = os.path.join(DRIVE_SAVE_DIR, "best.pt")
shutil.copy2(final_best, dest)

print(f"Model saved to Drive: {dest}")

## Step 9 — Retrain from best checkpoint (optional)

Run after adding more labeled images. Picks up from the best model automatically.

In [ ]:
from datetime import datetime

best_models = glob.glob(os.path.join(TRAINING_RUNS, "*/weights/best.pt"))
latest_best = max(best_models, key=os.path.getmtime)
run_name    = f"water_meter_retrain_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Retraining from: {latest_best}")

model = YOLO(latest_best)
model.train(
    data=DATA_YAML,
    epochs=20,
    imgsz=416,
    batch=32,
    workers=4,
    cache=True,
    amp=True,
    patience=10,
    optimizer="AdamW",
    cos_lr=True,
    project=TRAINING_RUNS,
    name=run_name,
)

print(f"\nRetrain complete. Run saved to: {TRAINING_RUNS}/{run_name}")